# 02 — Baselines: text-embedding models and contextual encoders

**What this notebook does.** It scores every (question, reference, candidate) triple of the benchmark with evaluators that *do not generate text*:

| Family | Evaluators | How the score is computed |
|---|---|---|
| Text-embedding models (>= 2) | BGE-M3, multilingual-E5-large, Qwen3-Embedding-0.6B | cosine similarity between the embedding of the reference and of the candidate |
| Contextual encoders (>= 2) | BERTScore with CAMeLBERT-mix (Arabic), MARBERTv2 (dialect-aware), XLM-R-large (multilingual) | greedy matching of token embeddings from one intermediate layer; we report F1 |

**Why two reference registers.** An evaluator compares a candidate with a *reference*. If the reference is Egyptian and the candidate is Egyptian, they share words, so the score is high for a trivial reason. To separate *"the candidate looks different from the reference"* from *"the evaluator dislikes this register"*, every candidate is scored against **both** an Egyptian reference and an MSA reference (`ref_register`). This gives 360 candidates x 2 = 720 scored pairs per evaluator.

**Design notes.**
* No fine-tuning and no prompt tuning: these are the standard off-the-shelf baselines.
* BERTScore layer choices follow the package defaults for comparable models (layer 9 for 12-layer BERTs, layer 17 for XLM-R-large). No IDF weighting and no baseline rescaling (rescaling constants do not exist for these encoders); all analysis later is normalised per evaluator, so raw scales do not matter.
* If one model fails to download or load, it is skipped with a message instead of stopping the notebook; re-run to retry.

In [ ]:
import os, sys
REPO_DIR = "register_bias_judge"
if not os.path.exists("src/common.py"):
    if os.path.isdir(REPO_DIR):                                   # unzipped next to the notebook
        os.chdir(REPO_DIR)
    else:
        try:                                                      # Google Colab: the repo lives in your Drive, so
            from google.colab import drive                        # data/ and results/ survive between sessions
            drive.mount("/content/drive")
            os.chdir(f"/content/drive/MyDrive/{REPO_DIR}")
        except Exception as e:
            print("Could not locate the repo folder:", e)
sys.path.insert(0, os.getcwd())
assert os.path.exists("src/common.py"), "Folder 'register_bias_judge' not found. Put it in the root of your Google Drive (My Drive) and re-run."
print("working in:", os.getcwd())

In [ ]:
%pip install -q "transformers>=4.51" sentence-transformers bert-score scipy

In [ ]:
import numpy as np, pandas as pd
from src import common as C
from src import models as M

C.set_seed(C.SEED)
print("device:", M.device())

EMBEDDING_MODELS = {   # name -> Hugging Face id and the text prefix the model expects
    "bge-m3":                dict(id="BAAI/bge-m3",                    prefix=""),
    "multilingual-e5-large": dict(id="intfloat/multilingual-e5-large", prefix="query: "),
    "qwen3-embedding-0.6b":  dict(id="Qwen/Qwen3-Embedding-0.6B",      prefix=""),
}
ENCODERS = {           # BERTScore backbones: name -> Hugging Face id and the layer used for token embeddings
    "bertscore-camelbert-mix": dict(id="CAMeL-Lab/bert-base-arabic-camelbert-mix", layer=9),
    "bertscore-marbertv2":     dict(id="UBC-NLP/MARBERTv2",                        layer=9),
    "bertscore-xlmr-large":    dict(id="xlm-roberta-large",                        layer=17),
}

## 1) Load the benchmark and build the scoring pairs

In [ ]:
items = C.load_items()          # 30 questions x 3 quality levels x 4 registers (+ human ratings / corrections if present)
pairs = C.build_pairs(items)    # x 2 reference registers
print(pairs.shape)
pairs.groupby(["ref_register", "level"]).size().unstack()

## 2) Text-embedding models: cosine similarity to the reference

In [ ]:
records = []
for name, cfg in EMBEDDING_MODELS.items():
    try:
        s = M.embedding_cosine(cfg["id"], pairs, prefix=cfg["prefix"])
        records.append(pd.DataFrame({"pair_id": pairs.pair_id, "evaluator": name, "family": "embedding", "score": s}))
        print(f"done: {name}")
    except Exception as e:
        print(f"[skipped] {name}: {type(e).__name__}: {str(e)[:200]}")

## 3) Contextual encoders: BERTScore F1

In [ ]:
for name, cfg in ENCODERS.items():
    try:
        s = M.bertscore_f1(cfg["id"], cfg["layer"], pairs.candidate.tolist(), pairs.reference.tolist())
        records.append(pd.DataFrame({"pair_id": pairs.pair_id, "evaluator": name, "family": "bertscore", "score": s}))
        print(f"done: {name}")
    except Exception as e:
        print(f"[skipped] {name}: {type(e).__name__}: {str(e)[:200]}")

## 4) Save and sanity-check

A useful evaluator must rank **correct > partial > incorrect**. If an evaluator does not, its register-bias numbers in notebook 04 are not meaningful (it is not measuring content at all), so we check this before moving on.

In [ ]:
scores = pd.concat(records, ignore_index=True)
print("saved:", C.save_scores(scores, "embedding_encoder"))

chk = scores.merge(pairs[["pair_id", "level"]], on="pair_id")
tab = chk.groupby(["evaluator", "level"]).score.mean().unstack()[C.LEVELS]
tab["separates_levels"] = (tab.correct > tab.partial) & (tab.partial > tab.incorrect)
tab.round(3)